## Реализация многослойного перцептрона

In [1]:
import numpy as np

### Слой прямого распространения

Класс слоя прямого распространения (аффинное преобразование: $WX+b$), где задаётся количество нейронов на слое.

In [2]:
class DenseLayer:
    def __init__(self, in_features, out_features):
        self.w =  np.random.randn(out_features, in_features) * 0.1
        self.b = np.zeros((out_features, 1))
        self.x_previous = None
        self.dw = None
        self.db = None

    def forward(self, x_previous):
        self.x_previous = x_previous
        z = np.dot(self.w, x_previous) + self.b
        return z
    
    def backward(self, sigma_mse):      # sigma_mse -> sigma'(z^(L)) * 2 * (a^(L) - y)
        if sigma_mse.ndim == 1:
            sigma_mse = sigma_mse.reshape(-1, 1)

        self.dw = np.dot(sigma_mse, self.x_previous.T)
        self.db = sigma_mse
        da_previous = np.dot(self.w.T, sigma_mse)
        return da_previous

Классы слоёв, представляющих собой функций активации: ReLU, Tanh, Sigmoid, достаточно одной.

In [3]:
class ReLU:
    def __init__(self):
        self.z = None

    def forward(self, z):
        self.z = z
        return np.maximum(0, z)
    
    def backward(self, da):
        # sigma_mse = []
        # for i in da:
        #     if i > 0: sigma_mse.append(i)
        #     else: sigma_mse.append(0)
        # return sigma_mse
        return da * (self.z > 0)

In [4]:
class Sigmoid:
    def __init__(self):
        self.a = None

    def forward(self, z):
        self.a = 1 / (1 + np.exp(-z))
        return self.a

    def backward(self, da):
        return da * (self.a * (1 - self.a))

Класс сети прямого распространения, принимающий список слоёв.

In [5]:
class Network:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    def backward(self, grad):  #grad -> 2 * (y_pred - y_true)
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

Класс оптимизатора (градиентный спуск). Т.е. способ оптимизации не вшит в сеть, он задаётся “снаружи”.

In [6]:
class SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate

    def step(self, network):
        for layer in network.layers:
            if hasattr(layer, 'w'):     # has attribute
                layer.w -= self.learning_rate * layer.dw
                layer.b -= self.learning_rate * layer.db

### Решение задачи регрессии

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [8]:
regression = pd.read_csv('../data/diamonds_filtered.csv')
y_reg = regression['price']
x_reg = regression.drop(['price'], axis=1)
x_train_reg, x_test_reg, y_train_reg, y_test_reg = train_test_split(x_reg, y_reg, test_size=0.15, random_state=81, shuffle=True)

scaler_reg = StandardScaler()
x_train_reg = scaler_reg.fit_transform(x_train_reg)
x_test_reg = scaler_reg.transform(x_test_reg)

In [9]:
layers_reg = [
    DenseLayer(in_features=len(x_reg.columns), out_features=32),
    ReLU(),
    DenseLayer(in_features=32, out_features=64),
    ReLU(),
    DenseLayer(in_features=64, out_features=1)
]

model_reg = Network(layers_reg)
optimizer = SGD(learning_rate=1e-8)

In [10]:
epochs = 50
r2_best = 0

for epoch in range(epochs):
    all_preds = []
    all_true = []
    
    for i in range(len(x_train_reg)):
        x = x_train_reg[i].reshape(-1, 1)
        y_val = y_train_reg.iloc[i]
        y_true = np.array([[y_val]])
        
        y_pred = model_reg.forward(x)
        all_preds.append(y_pred[0][0])
        all_true.append(y_val)
        
        loss_grad = 2 * (y_pred - y_true)
        model_reg.backward(loss_grad)
        optimizer.step(model_reg)

    r2 = r2_score(all_true, all_preds)
    r2_best = max(r2, r2_best)
    
    print(f"Epoch {epoch}: R2 = {round(r2, 4)}")

Epoch 0: R2 = -0.0107
Epoch 1: R2 = 0.9498
Epoch 2: R2 = 0.9592
Epoch 3: R2 = 0.9632
Epoch 4: R2 = 0.9658
Epoch 5: R2 = 0.9683
Epoch 6: R2 = 0.9696
Epoch 7: R2 = 0.9705
Epoch 8: R2 = 0.9713
Epoch 9: R2 = 0.9723
Epoch 10: R2 = 0.9733
Epoch 11: R2 = 0.9742
Epoch 12: R2 = 0.9749
Epoch 13: R2 = 0.9754
Epoch 14: R2 = 0.9758
Epoch 15: R2 = 0.9761
Epoch 16: R2 = 0.9764
Epoch 17: R2 = 0.9766
Epoch 18: R2 = 0.9767
Epoch 19: R2 = 0.9768
Epoch 20: R2 = 0.9769
Epoch 21: R2 = 0.977
Epoch 22: R2 = 0.9771
Epoch 23: R2 = 0.9772
Epoch 24: R2 = 0.9772
Epoch 25: R2 = 0.9773
Epoch 26: R2 = 0.9774
Epoch 27: R2 = 0.9775
Epoch 28: R2 = 0.9776
Epoch 29: R2 = 0.9776
Epoch 30: R2 = 0.9777
Epoch 31: R2 = 0.9779
Epoch 32: R2 = 0.9779
Epoch 33: R2 = 0.978
Epoch 34: R2 = 0.978
Epoch 35: R2 = 0.9781
Epoch 36: R2 = 0.9782
Epoch 37: R2 = 0.9782
Epoch 38: R2 = 0.9783
Epoch 39: R2 = 0.9783
Epoch 40: R2 = 0.9784
Epoch 41: R2 = 0.9784
Epoch 42: R2 = 0.9784
Epoch 43: R2 = 0.9785
Epoch 44: R2 = 0.9785
Epoch 45: R2 = 0.9785


In [11]:
results = []
results.append({"Task": "Regression", "Value": r2_best})

### Решение задачи классификации

In [12]:
from sklearn.metrics import f1_score
from imblearn.combine import SMOTEENN

In [13]:
classification = pd.read_csv('../data/credit_card_fraud.csv')
y_cl = classification['fraud']
x_cl = classification.drop('fraud', axis=1)
x_train_cl, x_test_cl, y_train_cl, y_test_cl = train_test_split(x_cl, y_cl, test_size=0.15, random_state=81, shuffle=True, stratify=y_cl)

scaler_cl = StandardScaler()
x_train_cl = scaler_cl.fit_transform(x_train_cl)
x_test_cl = scaler_cl.transform(x_test_cl)

In [14]:
sme = SMOTEENN(random_state=81, n_jobs=-1)
x_train_cl, y_train_cl = sme.fit_resample(x_train_cl, y_train_cl)

/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/imblearn/over_sampling/_smote/base.py:370: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(


In [15]:
layers_cl = [
    DenseLayer(in_features=len(x_cl.columns), out_features=32),
    ReLU(),
    DenseLayer(in_features=32, out_features=64),
    ReLU(),
    DenseLayer(in_features=64, out_features=1)
]

model_cl = Network(layers_cl)
optimizer = SGD(learning_rate=0.0001)

In [16]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

In [17]:
epochs = 5
f1_best = 0

for epoch in range(epochs):
    all_preds = []
    all_true = []
    
    for i in range(len(x_train_cl)):
        x = x_train_cl[i].reshape(-1, 1)
        y_true = y_train_cl.iloc[i]
        
        z = model_cl.forward(x)
        
        y_prob = sigmoid(z)
        all_preds.append(1 if y_prob > 0.5 else 0)
        all_true.append(y_true)

        loss_grad = y_prob - y_true
        
        model_cl.backward(loss_grad)
        optimizer.step(model_cl)

    f1 = f1_score(all_true, all_preds)
    f1_best = max(f1, f1_best)
    print(f"Epoch {epoch}: F1 = {round(f1, 4)}")

Epoch 0: F1 = 0.9904
Epoch 1: F1 = 0.9961
Epoch 2: F1 = 0.9976
Epoch 3: F1 = 0.9982
Epoch 4: F1 = 0.9984


In [18]:
results.append({"Task": "Classification", "Value": f1_best})

### Вывод

Самостоятельно реализованный многослойный перцептрон показывает те же метрики, что `MLPRegressor` и `MLPClassifier` из sklearn🔥

In [19]:
res_df = pd.DataFrame(results)
res_df

,Task,Value
0,Regression,0.978650
1,Classification,0.998428
